# Experiment: Simple Two-Sideband Processing Example

Minimal standalone workflow:

- choose a dataset folder
- choose a passage and harmonic
- run two-sideband opening
- show raw and processed amplitude/phase images


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().resolve()
assert (repo_root / "hologram_opening.py").exists(), "Start Jupyter from the repository root."
sys.path.insert(0, str(repo_root))

from hologram_opening import open_hologram_2d, processed_phase
from processing import load_passage

plt.rcParams["figure.figsize"] = (14, 10)
plt.rcParams["image.cmap"] = "viridis"
np.set_printoptions(precision=4, suppress=True)

print(f"Using repo root: {repo_root}")


In [ ]:
folder_path = ""
passage = "forward"
harmonic_index = 4

assert folder_path.strip(), "Set folder_path to your dataset directory before running this cell."
assert passage in {"forward", "reverse"}, "passage must be 'forward' or 'reverse'"
assert 0 <= harmonic_index <= 5, "harmonic_index must be between 0 and 5"


In [ ]:
loaded = load_passage(folder_path, passage=passage, processing_mode="two_sideband")
raw_hologram = loaded.o[:, :, harmonic_index]

stage_images, carrier_row, filter_width_y, diagnostics = open_hologram_2d(
    raw_hologram,
    processing_mode="two_sideband",
)

processed = stage_images["processed"]
raw_amplitude = np.abs(raw_hologram)
raw_phase = np.angle(raw_hologram)
processed_amplitude = np.abs(processed)
processed_phase_image = processed_phase(processed, processing_mode="two_sideband")

summary = {
    "folder_path": folder_path,
    "image_name": loaded.image_name,
    "passage": passage,
    "harmonic_index": harmonic_index,
    "shape": raw_hologram.shape,
    "carrier_row": carrier_row,
    "filter_width_y": filter_width_y,
    **diagnostics,
}
summary


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
plots = [
    (raw_amplitude, "Raw amplitude", "hot"),
    (processed_amplitude, "Processed amplitude", "hot"),
    (raw_phase, "Raw phase", "twilight"),
    (processed_phase_image, "Processed phase", "bwr"),
]

for axis, (image, title, cmap) in zip(axes.ravel(), plots):
    im = axis.imshow(image, aspect="auto", cmap=cmap)
    axis.set_title(title)
    axis.set_xlabel("X")
    axis.set_ylabel("Y")
    fig.colorbar(im, ax=axis, fraction=0.046, pad=0.04)

plt.show()


In [ ]:
data = {
    "raw_hologram": raw_hologram,
    "processed": processed,
    "raw_amplitude": raw_amplitude,
    "raw_phase": raw_phase,
    "processed_amplitude": processed_amplitude,
    "processed_phase": processed_phase_image,
}
print({key: value.shape for key, value in data.items()})
